# The Colorado River: A System Under Strain

Seven states and Mexico share water that starts as snow in the Rocky Mountains,
travels a thousand miles through two of the country's largest reservoirs, and
has been promised out faster than the river delivers it. This notebook tells
that story in the river's own data: the snowpack that feeds it, the runoff it
produces, the timing of its flow, and the reservoirs that used to be able to
absorb a bad year but increasingly can't.

In [ ]:
from __future__ import annotations

import logging

from IPython.display import Markdown, display

from colorado_river_viz.cache import load_snotel_stations, refresh_story
from colorado_river_viz.charts.ch1_supply import (
    build_paleo_context,
    build_supply_vs_compact,
)
from colorado_river_viz.charts.ch2_snow import build_snow_spaghetti
from colorado_river_viz.charts.ch3_runoff import (
    build_efficiency_trend,
    build_snow_vs_runoff,
    residual_efficiency_trend,
)
from colorado_river_viz.charts.ch4_timing import (
    build_cisco_spaghetti,
    build_timing_trends,
)
from colorado_river_viz.charts.ch5_dams import build_before_after_dam
from colorado_river_viz.charts.ch6_reservoirs import build_reservoir_storage
from colorado_river_viz.metrics.trend import theil_sen_trend
from colorado_river_viz.narrative import (
    describe_dam_effect,
    describe_peak_swe,
    describe_reservoir_drawdown,
    describe_runoff_year,
    describe_supply_vs_compact,
    describe_timing_trend,
    describe_trend,
)
from colorado_river_viz.settings import Settings
from colorado_river_viz.story_tables import (
    annual_supply,
    cisco_hydrograph,
    lees_ferry_regimes,
    paleo_supply,
    reservoir_storage,
    runoff_vs_snow,
    snow_annual,
    snow_index_daily,
    snow_index_envelope,
    timing_annual,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

REFRESH = "offline"  # "incremental" or "full" to hit the live APIs instead

settings = Settings()
refresh_story(REFRESH, settings)
stations = load_snotel_stations(settings.cache_dir)

## Meet the river

*(map -- T24)*

## 1 · Promised more than it has

In 1922, seven states divided the Colorado River's flow before anyone had
measured it through a full wet-dry cycle. They allocated 16.5 million
acre-feet a year -- 7.5 MAF each to the upper and lower basins, plus 1.5 MAF
promised to Mexico in 1944. The river has rarely delivered that much.

In [ ]:
supply = annual_supply(settings.cache_dir)
paleo = paleo_supply(settings.cache_dir)

fig_supply = build_supply_vs_compact(supply)
fig_supply.show()

fig_paleo = build_paleo_context(paleo, supply)
fig_paleo.show()

In [ ]:
display(Markdown(f"**{describe_supply_vs_compact(supply)}**"))

**How we know:** Natural flow is Reclamation's naturalized Lees Ferry flow,
WY1906 onward, published and periodically revised; provisional years may still
change. WY2025-26 aren't in the published file yet, so they're estimated from
Powell unregulated inflow (RISE 512) via a linear fit over WY1964-2020 (the
file's final years) and shown as hatched bars with an error whisker (±1.96 x
the fit's residual SD; decision 0014). The paleo context splices Meko et al.
(2007)'s tree-ring reconstruction (762-2005) onto the same gauge/estimate
series used above, so the 20-year rolling mean runs unbroken to the present.
Sources: Bureau of Reclamation naturalized flow; treeflow.info (Meko et al.
2007, *Geophysical Research Letters* 34, L10705); USGS/RISE Powell unregulated
inflow.

## 2 · It starts as snow

Every drop of Colorado River water begins as snow in the mountains of Colorado,
Wyoming, Utah, and New Mexico. A fixed set of 54 long-record SNOTEL stations
(decision 0013) tracks the basin's snow water equivalent (SWE) -- the depth of
water the snowpack would produce if it melted all at once -- every winter back
to the early 1980s.

In [ ]:
snow_daily = snow_index_daily(settings.cache_dir, stations)
snow_envelope = snow_index_envelope(snow_daily)
snow_year = snow_annual(settings.cache_dir, stations, snow_daily)

fig_snow = build_snow_spaghetti(snow_daily, snow_envelope)
fig_snow.show()

In [ ]:
latest_snow = snow_year.dropna(subset=["peak_swe_in"]).iloc[-1]
display(Markdown(f"**{describe_peak_swe(latest_snow)}**"))

**How we know:** Basin SWE is the mean across the 54 index stations reporting
that day, requiring at least 90% of the index to report (decision 0017); gaps
of 7 days or less are linearly interpolated first. `pct_of_median` compares the
sum of reporting stations' SWE against the sum of their own 1991-2020 NRCS
medians -- the standard basin-index method. Source: NRCS SNOTEL, `AWDB` API.

## 3 · Same snow, less river

A given amount of mountain snowpack no longer turns into the same amount of
spring runoff it once did. Warmer soils and thirstier vegetation intercept more
of the melt before it reaches a gauge, so the relationship between peak SWE and
the Apr-Jul runoff pulse at Lake Powell has been sliding for decades.

In [ ]:
runoff = runoff_vs_snow(settings.cache_dir, snow_year)

fig_scatter = build_snow_vs_runoff(runoff)
fig_scatter.show()

fig_trend = build_efficiency_trend(runoff)
fig_trend.show()

In [ ]:
latest_runoff = runoff.iloc[-1]
efficiency_trend = theil_sen_trend(runoff["water_year"], runoff["runoff_efficiency"])
display(Markdown(f"**{describe_runoff_year(latest_runoff)}**"))
display(Markdown(describe_trend(efficiency_trend, "Spring runoff efficiency")))

In [ ]:
# Robustness check (T16 carry-over note, decision 0025): a run of dry years can
# make runoff_efficiency look like it's declining on its own, since base flow
# doesn't shrink with snow. The trend in the *residuals* of a runoff-vs-peak-SWE
# regression isn't vulnerable to that, so we check it agrees in sign and
# significance before trusting the headline -- see decision 0025 for why the
# comparison is on tau/p rather than on pct_of_normal_per_decade.
robustness_trend = residual_efficiency_trend(runoff)
print(
    f"direct:  slope/decade={efficiency_trend.slope_per_decade:.3f}  "
    f"tau={efficiency_trend.kendall_tau:.2f}  p={efficiency_trend.p_value:.3f}"
)
print(
    f"resid.:  slope/decade={robustness_trend.slope_per_decade:.3f}  "
    f"tau={robustness_trend.kendall_tau:.2f}  p={robustness_trend.p_value:.3f}"
)
trends_agree = (
    efficiency_trend.slope_per_decade < 0 and robustness_trend.slope_per_decade < 0
)
assert trends_agree, (
    "the direct and residual trends disagree in sign -- revisit the chapter 3 narrative"
)

**How we know:** `runoff_efficiency` is Apr-Jul unregulated inflow at Lake
Powell (RISE item 512) in MAF, divided by that water year's peak basin SWE in
inches (decision 0017). The Theil-Sen trend above is robust to outliers, and
Kendall's tau tests whether the decline is more than noise. Because a run of
dry years alone can inflate this ratio's apparent decline (base flow doesn't
shrink with snow the way peak flow does), we also checked the trend in the
residuals of a runoff-vs-peak-SWE regression -- a check that isn't vulnerable to
that effect. Both trends are negative and both are statistically significant
(see the printed comparison above; decision 0025 has the numbers from the real
cache and explains why we compare the two trends' sign and Kendall's tau rather
than their `pct_of_normal_per_decade`, which isn't meaningful for a
near-zero-mean residual series). Source: USGS/RISE Powell unregulated inflow
and the SNOTEL index above.

## 4 · Earlier and faster

Snowmelt no longer waits for summer. Warmer springs pull the peak snowpack,
the melt-out, and the resulting streamflow earlier in the year than they ran a
century ago -- and once the water starts moving, it moves through faster.

In [ ]:
timing = timing_annual(settings.cache_dir, snow_year)
hydrograph = cisco_hydrograph(settings.cache_dir)

fig_timing = build_timing_trends(timing)
fig_timing.show()

fig_hydrograph = build_cisco_spaghetti(hydrograph)
fig_hydrograph.show()

In [ ]:
cov_trend = theil_sen_trend(
    timing.dropna(subset=["cisco_center_of_volume_doy"])["water_year"],
    timing.dropna(subset=["cisco_center_of_volume_doy"])["cisco_center_of_volume_doy"],
)
display(Markdown(describe_timing_trend(cov_trend, "The Cisco half-flow date")))

**How we know:** Snow peak and melt-out day are the per-station median day of
water year (decision 0024), from the same fixed SNOTEL index as chapters 2-3.
Cisco's center-of-volume day is the day of water year by which half the
year's flow has passed, counted only in complete water years. The USGS Cisco
record has no data for WY1918-1922, so the pre-dam baseline envelope
(1914-1962) draws on 44 complete years, not 49. Source: USGS gauge
09180500 (Colorado River near Cisco, UT).

## 5 · Dams flatten the river

Before Glen Canyon Dam, Lees Ferry's flow followed the snowmelt: a sharp spring
peak, then a long summer and winter recession. The dam turned that river into
a reservoir release, timed to power demand and downstream deliveries instead
of the snowpack. The seasonal pulse that shaped the canyon for millennia is
now a managed, nearly flat line.

In [ ]:
regimes = lees_ferry_regimes(settings.cache_dir)

fig_dams = build_before_after_dam(regimes)
fig_dams.show()

In [ ]:
display(Markdown(f"**{describe_dam_effect(regimes.annual_peaks)}**"))

**How we know:** `before_dam` is water year 1922-1962 (pre-dam, USGS gauge 09380000 at Lees Ferry); `after_dam` is 1981 onward, once Lake Powell first filled, so the 1963-1980 filling years don't blur the comparison. Lines are the per-day-of-water-year median flow in each regime, with a 10th-90th percentile band; the takeaway sentence compares each regime's mean annual peak-day flow.

## 6 · The bank account

Lake Powell and Lake Mead exist to smooth the difference between a variable
river and steady promises: they hold water in wet years to cover dry ones.
That buffer has been drawn down for a quarter century, as demand and a drying
climate have outpaced what the river delivers -- turning a bank account built
for occasional shortfalls into one running low year after year.

In [ ]:
storage = reservoir_storage(settings.cache_dir)

fig_storage = build_reservoir_storage(storage)
fig_storage.show()

In [ ]:
display(Markdown(f"**{describe_reservoir_drawdown(storage)}**"))

**How we know:** `pct_full` is RISE storage divided by Reclamation's operational full-pool capacity (24.3 MAF for Lake Powell, 26.12 MAF for Lake Mead) -- the same convention behind the public "percent full" figures, not the higher capacities from recent topobathymetric resurveys. `ft_above_min_power_pool` compares by elevation rather than storage, since Lake Powell's RISE storage runs 2-8% above its 2017 area-capacity table at a given elevation. Combined storage covers days both reservoirs report, starting in 1964 when Powell storage begins. Source: Reclamation RISE (reservoir elevation and storage), reservoirs.py.

## 2026 at a glance

*(KPI panel -- T25)*

## Sources and methods

*(T26)*